In [20]:
import pandas as pd

In [21]:
# file path
file_path = "ProductionConsumptionSettlement.csv"

# load csv (important: separator is ;)
pcs = pd.read_csv(file_path, sep=";")

# basic preview
print("Shape:", pcs.shape)
print("\nColumns:")
print(pcs.columns.tolist())

print("\nFirst 5 rows:")
display(pcs.head())

Shape: (26304, 28)

Columns:
['HourUTC', 'HourDK', 'PriceArea', 'CentralPowerMWh', 'LocalPowerMWh', 'CommercialPowerMWh', 'LocalPowerSelfConMWh', 'OffshoreWindLt100MW_MWh', 'OffshoreWindGe100MW_MWh', 'OnshoreWindLt50kW_MWh', 'OnshoreWindGe50kW_MWh', 'HydroPowerMWh', 'SolarPowerLt10kW_MWh', 'SolarPowerGe10Lt40kW_MWh', 'SolarPowerGe40kW_MWh', 'SolarPowerSelfConMWh', 'UnknownProdMWh', 'ExchangeNO_MWh', 'ExchangeSE_MWh', 'ExchangeGE_MWh', 'ExchangeNL_MWh', 'ExchangeGB_MWh', 'ExchangeGreatBelt_MWh', 'GrossConsumptionMWh', 'GridLossTransmissionMWh', 'GridLossInterconnectorsMWh', 'GridLossDistributionMWh', 'PowerToHeatMWh']

First 5 rows:


,HourUTC,HourDK,PriceArea,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,LocalPowerSelfConMWh,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,...,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeNL_MWh,ExchangeGB_MWh,ExchangeGreatBelt_MWh,GrossConsumptionMWh,GridLossTransmissionMWh,GridLossInterconnectorsMWh,GridLossDistributionMWh,PowerToHeatMWh
0,2022-01-01 00:00:00,2022-01-01 01:00:00,DK1,"293,093959","156,007948","85,381966","16,197998","138,357348","899,588499","6,556590",...,"403,613000","570,140000","-583,754000",NaN,"356,500000","2206,210314","73,389710","58,017700","87,579944","198,883385"
1,2022-01-01 01:00:00,2022-01-01 02:00:00,DK1,"384,846011","152,127928","86,865465","15,207601","119,702698","921,140135","5,398335",...,"396,447000","564,190000","-554,725000",NaN,"293,300000","2147,758315","71,933507","56,611600","85,348963","199,159845"
2,2022-01-01 02:00:00,2022-01-01 03:00:00,DK1,"365,706619","154,854604","86,807320","15,404626","119,564946","835,099673","4,786685",...,"295,690000","453,260000","-237,741000",NaN,"272,800000","2023,389045","67,586565","45,209300","82,920562","136,606848"
3,2022-01-01 03:00:00,2022-01-01 04:00:00,DK1,"379,162111","154,907694","86,882696","15,542090","125,643131","678,922773","2,741542",...,"306,195000","111,260000","341,888000",NaN,"196,300000","2050,223766","62,396313","46,983300","82,326682","205,360082"
4,2022-01-01 04:00:00,2022-01-01 05:00:00,DK1,"370,112134","156,512416","87,018405","14,962279","157,398152","736,866967","2,252833",...,"167,488000","127,210000","382,106000",NaN,"380,400000","2065,765754","73,352255","45,350000","79,815472","214,499309"


In [22]:
#Check Price Areas
# check unique price areas
print("Unique Price Areas:")
print(pcs["PriceArea"].unique())

# count rows per price area
print("\nRows per Price Area:")
print(pcs["PriceArea"].value_counts())

Unique Price Areas:
<StringArray>
['DK1']
Length: 1, dtype: str

Rows per Price Area:
PriceArea
DK1    26304
Name: count, dtype: int64


This means:

The dataset contains only DK1

No DK2 data mixed inside

This is exactly what  thesis requires, because the proposal clearly restricts the analysis to the DK1 bidding zone to keep the system homogeneous.

So no filtering is needed here.

Total expected hours
8760
+
8760
+
8784
=
26304
8760+8760+8784=26304

dataset has exactly the correct number of rows.

In [23]:
#Clean Time Columns
# Remove Danish local time column
pcs = pcs.drop(columns=["HourDK"])

# Convert HourUTC to datetime
pcs["HourUTC"] = pd.to_datetime(pcs["HourUTC"])

#set_time_stamp_to_match_with_oters_data
pcs["HourUTC"] = pcs["HourUTC"].dt.tz_localize("UTC")

# Check result
print(pcs.dtypes)

print("\nFirst timestamps:")
print(pcs["HourUTC"].head())

HourUTC                       datetime64[us, UTC]
PriceArea                                     str
CentralPowerMWh                               str
LocalPowerMWh                                 str
CommercialPowerMWh                            str
LocalPowerSelfConMWh                          str
OffshoreWindLt100MW_MWh                       str
OffshoreWindGe100MW_MWh                       str
OnshoreWindLt50kW_MWh                         str
OnshoreWindGe50kW_MWh                         str
HydroPowerMWh                                 str
SolarPowerLt10kW_MWh                          str
SolarPowerGe10Lt40kW_MWh                      str
SolarPowerGe40kW_MWh                          str
SolarPowerSelfConMWh                          str
UnknownProdMWh                                str
ExchangeNO_MWh                                str
ExchangeSE_MWh                                str
ExchangeGE_MWh                                str
ExchangeNL_MWh                                str


In [24]:
#Convert All Numeric Columns Properly
# Identify numeric columns (everything except HourUTC and PriceArea)
numeric_cols = pcs.columns.difference(["HourUTC", "PriceArea"])

# Replace comma with dot and convert to float
pcs[numeric_cols] = pcs[numeric_cols].replace(",", ".", regex=True).astype(float)

# Check datatypes again
print(pcs.dtypes)

HourUTC                       datetime64[us, UTC]
PriceArea                                     str
CentralPowerMWh                           float64
LocalPowerMWh                             float64
CommercialPowerMWh                        float64
LocalPowerSelfConMWh                      float64
OffshoreWindLt100MW_MWh                   float64
OffshoreWindGe100MW_MWh                   float64
OnshoreWindLt50kW_MWh                     float64
OnshoreWindGe50kW_MWh                     float64
HydroPowerMWh                             float64
SolarPowerLt10kW_MWh                      float64
SolarPowerGe10Lt40kW_MWh                  float64
SolarPowerGe40kW_MWh                      float64
SolarPowerSelfConMWh                      float64
UnknownProdMWh                            float64
ExchangeNO_MWh                            float64
ExchangeSE_MWh                            float64
ExchangeGE_MWh                            float64
ExchangeNL_MWh                            float64


In [25]:
#Verify Time Series Integrity
# Sort dataset by time
pcs = pcs.sort_values("HourUTC").reset_index(drop=True)

# Check duplicates
duplicate_times = pcs["HourUTC"].duplicated().sum()

# Check time difference between rows
time_diff = pcs["HourUTC"].diff().value_counts()

print("Duplicate timestamps:", duplicate_times)

print("\nTime differences between rows:")
print(time_diff.head())

Duplicate timestamps: 0

Time differences between rows:
HourUTC
0 days 01:00:00    26303
Name: count, dtype: int64


means:

Every observation is exactly 1 hour apart

Total intervals = 26303

Which is correct because:

26304
 rows
−
1
=
26303
 time intervals
26304 rows−1=26303 time intervals

So the dataset is perfectly continuous hourly data from 2022–2024 (including leap year 2024).

In [26]:
# Rename timestamp column to match other datasets
pcs = pcs.rename(columns={"HourUTC": "datetime_utc"})

# Drop PriceArea (since only DK1)
pcs = pcs.drop(columns=["PriceArea"])

# Check result
print(pcs.columns)
print("\nFirst rows:")
display(pcs.head())

Index(['datetime_utc', 'CentralPowerMWh', 'LocalPowerMWh',
       'CommercialPowerMWh', 'LocalPowerSelfConMWh', 'OffshoreWindLt100MW_MWh',
       'OffshoreWindGe100MW_MWh', 'OnshoreWindLt50kW_MWh',
       'OnshoreWindGe50kW_MWh', 'HydroPowerMWh', 'SolarPowerLt10kW_MWh',
       'SolarPowerGe10Lt40kW_MWh', 'SolarPowerGe40kW_MWh',
       'SolarPowerSelfConMWh', 'UnknownProdMWh', 'ExchangeNO_MWh',
       'ExchangeSE_MWh', 'ExchangeGE_MWh', 'ExchangeNL_MWh', 'ExchangeGB_MWh',
       'ExchangeGreatBelt_MWh', 'GrossConsumptionMWh',
       'GridLossTransmissionMWh', 'GridLossInterconnectorsMWh',
       'GridLossDistributionMWh', 'PowerToHeatMWh'],
      dtype='str')

First rows:


,datetime_utc,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,LocalPowerSelfConMWh,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,OnshoreWindGe50kW_MWh,HydroPowerMWh,...,ExchangeSE_MWh,ExchangeGE_MWh,ExchangeNL_MWh,ExchangeGB_MWh,ExchangeGreatBelt_MWh,GrossConsumptionMWh,GridLossTransmissionMWh,GridLossInterconnectorsMWh,GridLossDistributionMWh,PowerToHeatMWh
0,2022-01-01 00:00:00+00:00,293.093959,156.007948,85.381966,16.197998,138.357348,899.588499,6.556590,995.135115,1.641966,...,403.613,570.14,-583.754,NaN,356.5,2206.210314,73.389710,58.0177,87.579944,198.883385
1,2022-01-01 01:00:00+00:00,384.846011,152.127928,86.865465,15.207601,119.702698,921.140135,5.398335,893.822227,1.637933,...,396.447,564.19,-554.725,NaN,293.3,2147.758315,71.933507,56.6116,85.348963,199.159845
2,2022-01-01 02:00:00+00:00,365.706619,154.854604,86.807320,15.404626,119.564946,835.099673,4.786685,787.981293,1.639639,...,295.690,453.26,-237.741,NaN,272.8,2023.389045,67.586565,45.2093,82.920562,136.606848
3,2022-01-01 03:00:00+00:00,379.162111,154.907694,86.882696,15.542090,125.643131,678.922773,2.741542,781.776899,1.634412,...,306.195,111.26,341.888,NaN,196.3,2050.223766,62.396313,46.9833,82.326682,205.360082
4,2022-01-01 04:00:00+00:00,370.112134,156.512416,87.018405,14.962279,157.398152,736.866967,2.252833,614.282126,1.630415,...,167.488,127.21,382.106,NaN,380.4,2065.765754,73.352255,45.3500,79.815472,214.499309


In [27]:
#Missing Value Analysis
# Check missing values
missing_values = pcs.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

Missing values per column:
datetime_utc                      0
CentralPowerMWh                   0
LocalPowerMWh                     0
CommercialPowerMWh                0
LocalPowerSelfConMWh              0
OffshoreWindLt100MW_MWh           0
OffshoreWindGe100MW_MWh           0
OnshoreWindLt50kW_MWh             0
OnshoreWindGe50kW_MWh             0
HydroPowerMWh                     0
SolarPowerLt10kW_MWh              0
SolarPowerGe10Lt40kW_MWh          0
SolarPowerGe40kW_MWh              0
SolarPowerSelfConMWh              0
UnknownProdMWh                    0
ExchangeNO_MWh                    0
ExchangeSE_MWh                    0
ExchangeGE_MWh                    0
ExchangeNL_MWh                    0
ExchangeGB_MWh                16343
ExchangeGreatBelt_MWh             0
GrossConsumptionMWh               0
GridLossTransmissionMWh           0
GridLossInterconnectorsMWh        0
GridLossDistributionMWh           0
PowerToHeatMWh                    0
dtype: int64

Total missing values: 1

In [28]:
#first_non_missing_row
pcs[pcs["ExchangeGB_MWh"].notna()][["datetime_utc","ExchangeGB_MWh"]].head()

,datetime_utc,ExchangeGB_MWh
16343,2023-11-12 23:00:00+00:00,0.0
16344,2023-11-13 00:00:00+00:00,0.0
16345,2023-11-13 01:00:00+00:00,0.0
16346,2023-11-13 02:00:00+00:00,0.0
16347,2023-11-13 03:00:00+00:00,0.0


Energinet started recording the column only around Nov 2023.Before that, values are simply not reported, which appears as NaN.
This is typical when a new interconnector becomes relevant in the system.

In [29]:
#convert all missing values to zero.
pcs["ExchangeGB_MWh"] = pcs["ExchangeGB_MWh"].fillna(0)

# verify again
print("Remaining missing values:", pcs.isnull().sum().sum())

Remaining missing values: 0


Electricity exchange between Denmark and Great Britain is represented by the ExchangeGB_MWh variable. However, the Viking Link interconnector became operational only toward the end of the study period, resulting in missing values in earlier observations. These were interpreted as zero flows because no physical interconnection existed during those hours.

#Feature engineering



In [30]:
#Create Renewable Energy Features
# Total offshore wind
pcs["WindOffshore_MWh"] = (
    pcs["OffshoreWindLt100MW_MWh"] +
    pcs["OffshoreWindGe100MW_MWh"]
)

# Total onshore wind
pcs["WindOnshore_MWh"] = (
    pcs["OnshoreWindLt50kW_MWh"] +
    pcs["OnshoreWindGe50kW_MWh"]
)

# Total wind generation
pcs["TotalWind_MWh"] = pcs["WindOffshore_MWh"] + pcs["WindOnshore_MWh"]

# Total solar generation
pcs["TotalSolar_MWh"] = (
    pcs["SolarPowerLt10kW_MWh"] +
    pcs["SolarPowerGe10Lt40kW_MWh"] +
    pcs["SolarPowerGe40kW_MWh"] +
    pcs["SolarPowerSelfConMWh"]
)

# Total renewable generation
pcs["TotalRenewables_MWh"] = (
    pcs["TotalWind_MWh"] +
    pcs["TotalSolar_MWh"] +
    pcs["HydroPowerMWh"]
)

# check new columns
pcs[[
    "WindOffshore_MWh",
    "WindOnshore_MWh",
    "TotalWind_MWh",
    "TotalSolar_MWh",
    "TotalRenewables_MWh"
]].head()

,WindOffshore_MWh,WindOnshore_MWh,TotalWind_MWh,TotalSolar_MWh,TotalRenewables_MWh
0,1037.945847,1001.691705,2039.637552,0.021625,2041.301143
1,1040.842833,899.220562,1940.063395,0.022582,1941.723910
2,954.664619,792.767978,1747.432597,0.019940,1749.092176
3,804.565904,784.518441,1589.084345,0.022718,1590.741475
4,894.265119,616.534959,1510.800078,0.021027,1512.451520


Data Validation

soar value need to check as seems value is very small ,it may happen as olar generation is:

zero at night

low in winter

high in summer daytime

In [31]:
#check_solar
pcs["TotalSolar_MWh"].describe()

count    26304.000000
mean       267.576729
std        449.615233
min          0.013321
25%          0.218813
50%          8.397934
75%        363.194550
max       2306.786410
Name: TotalSolar_MWh, dtype: float64

Solar generation shows strong seasonal and diurnal patterns, with near-zero values during nighttime and higher values during summer daytime hours.

In [32]:
#check_total_solar
pcs[["datetime_utc", "TotalSolar_MWh"]].sort_values("TotalSolar_MWh", ascending=False).head()

,datetime_utc,TotalSolar_MWh
20819,2024-05-17 11:00:00+00:00,2306.786410
20914,2024-05-21 10:00:00+00:00,2275.247121
20818,2024-05-17 10:00:00+00:00,2273.877577
20484,2024-05-03 12:00:00+00:00,2272.768804
20820,2024-05-17 12:00:00+00:00,2260.897102


In [33]:
#Check Min / Max Values (Important EDA)
# Check summary statistics for key variables
cols_to_check = [
    "TotalSolar_MWh",
    "TotalWind_MWh",
    "TotalRenewables_MWh",
    "GrossConsumptionMWh",
    "CentralPowerMWh",
    "ExchangeNO_MWh"
]

pcs[cols_to_check].describe()

,TotalSolar_MWh,TotalWind_MWh,TotalRenewables_MWh,GrossConsumptionMWh,CentralPowerMWh,ExchangeNO_MWh
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000
mean,267.576729,1644.156421,1913.905374,2631.678200,469.182728,420.209688
std,449.615233,1232.256129,1248.781256,501.507219,344.156544,1047.778721
min,0.013321,0.825647,5.514189,1466.739560,1.285250,-1706.507896
25%,0.218813,546.962742,853.447270,2247.613476,223.978696,-380.545350
50%,8.397934,1424.996020,1764.077563,2645.197192,360.328916,687.167350
75%,363.194550,2551.633565,2800.702262,2991.146360,642.924470,1375.430775
max,2306.786410,4905.629931,6082.069750,4228.427907,1895.548334,1664.666000


In [34]:
#Check If Any Negative Values Exist
# Check negative values
negative_check = (pcs.select_dtypes(include=["float64"]) < 0).sum()

print("Negative values per column:")
print(negative_check[negative_check > 0])

Negative values per column:
ExchangeNO_MWh            8866
ExchangeSE_MWh            7793
ExchangeGE_MWh           17342
ExchangeNL_MWh           14833
ExchangeGB_MWh            6006
ExchangeGreatBelt_MWh    13361
dtype: int64


The dataset was validated through summary statistics and sanity checks to ensure that all variables were within realistic physical ranges and contained no negative or implausible values.

In [35]:
# 1. Total conventional generation
pcs["TotalConventional_MWh"] = (
    pcs["CentralPowerMWh"] +
    pcs["LocalPowerMWh"] +
    pcs["CommercialPowerMWh"] +
    pcs["LocalPowerSelfConMWh"] +
    pcs["UnknownProdMWh"]
)

# 2. Net exchange across all interconnectors
exchange_cols = [
    "ExchangeNO_MWh",
    "ExchangeSE_MWh",
    "ExchangeGE_MWh",
    "ExchangeNL_MWh",
    "ExchangeGB_MWh",
    "ExchangeGreatBelt_MWh"
]

pcs["NetExchange_MWh"] = pcs[exchange_cols].sum(axis=1)

# 3. Total imports = sum of positive exchange values (import is possitive values)
pcs["TotalImports_MWh"] = pcs[exchange_cols].clip(lower=0).sum(axis=1)

# 4. Total exports = absolute sum of negative exchange values (export is negetive values)
pcs["TotalExports_MWh"] = (-pcs[exchange_cols].clip(upper=0)).sum(axis=1)

pcs[[
    "datetime_utc",
    "TotalConventional_MWh",
    "NetExchange_MWh",
    "TotalImports_MWh",
    "TotalExports_MWh"
]].head()

,datetime_utc,TotalConventional_MWh,NetExchange_MWh,TotalImports_MWh,TotalExports_MWh
0,2022-01-01 00:00:00+00:00,550.681871,-385.7727,1330.253,1716.0257
1,2022-01-01 01:00:00+00:00,639.047005,-433.0126,1253.937,1686.9496
2,2022-01-01 02:00:00+00:00,622.773169,-348.4763,1021.750,1370.2263
3,2022-01-01 03:00:00+00:00,636.494591,-177.0123,955.643,1132.6553
4,2022-01-01 04:00:00+00:00,628.605234,-75.2910,1057.204,1132.4950


output looks internally consistent

A negative NetExchange_MWh means:

exports were larger than imports in that hour

A positive NetExchange_MWh would mean:

imports were larger than exports

So these new variables make sense physically.

In [36]:
#creating TotalProduction_MWh feature
pcs["TotalProduction_MWh"] = (
    pcs["TotalConventional_MWh"] +
    pcs["TotalRenewables_MWh"]
)

pcs[[
    "datetime_utc",
    "TotalProduction_MWh",
    "GrossConsumptionMWh",
    "TotalRenewables_MWh",
    "TotalConventional_MWh"
]].head()

,datetime_utc,TotalProduction_MWh,GrossConsumptionMWh,TotalRenewables_MWh,TotalConventional_MWh
0,2022-01-01 00:00:00+00:00,2591.983014,2206.210314,2041.301143,550.681871
1,2022-01-01 01:00:00+00:00,2580.770915,2147.758315,1941.723910,639.047005
2,2022-01-01 02:00:00+00:00,2371.865345,2023.389045,1749.092176,622.773169
3,2022-01-01 03:00:00+00:00,2227.236066,2050.223766,1590.741475,636.494591
4,2022-01-01 04:00:00+00:00,2141.056754,2065.765754,1512.451520,628.605234


This helps us analyze:

Production vs Consumption

Later you can even create:

EnergyBalance = TotalProduction - GrossConsumption

This connects directly to:

imports (if deficit)

exports (if surplus)

In [37]:
print("Shape:", pcs.shape)
print("\nColumns:")
print(pcs.columns.tolist())

print("\nFirst 5 rows:")
display(pcs.head())

Shape: (26304, 36)

Columns:
['datetime_utc', 'CentralPowerMWh', 'LocalPowerMWh', 'CommercialPowerMWh', 'LocalPowerSelfConMWh', 'OffshoreWindLt100MW_MWh', 'OffshoreWindGe100MW_MWh', 'OnshoreWindLt50kW_MWh', 'OnshoreWindGe50kW_MWh', 'HydroPowerMWh', 'SolarPowerLt10kW_MWh', 'SolarPowerGe10Lt40kW_MWh', 'SolarPowerGe40kW_MWh', 'SolarPowerSelfConMWh', 'UnknownProdMWh', 'ExchangeNO_MWh', 'ExchangeSE_MWh', 'ExchangeGE_MWh', 'ExchangeNL_MWh', 'ExchangeGB_MWh', 'ExchangeGreatBelt_MWh', 'GrossConsumptionMWh', 'GridLossTransmissionMWh', 'GridLossInterconnectorsMWh', 'GridLossDistributionMWh', 'PowerToHeatMWh', 'WindOffshore_MWh', 'WindOnshore_MWh', 'TotalWind_MWh', 'TotalSolar_MWh', 'TotalRenewables_MWh', 'TotalConventional_MWh', 'NetExchange_MWh', 'TotalImports_MWh', 'TotalExports_MWh', 'TotalProduction_MWh']

First 5 rows:


,datetime_utc,CentralPowerMWh,LocalPowerMWh,CommercialPowerMWh,LocalPowerSelfConMWh,OffshoreWindLt100MW_MWh,OffshoreWindGe100MW_MWh,OnshoreWindLt50kW_MWh,OnshoreWindGe50kW_MWh,HydroPowerMWh,...,WindOffshore_MWh,WindOnshore_MWh,TotalWind_MWh,TotalSolar_MWh,TotalRenewables_MWh,TotalConventional_MWh,NetExchange_MWh,TotalImports_MWh,TotalExports_MWh,TotalProduction_MWh
0,2022-01-01 00:00:00+00:00,293.093959,156.007948,85.381966,16.197998,138.357348,899.588499,6.556590,995.135115,1.641966,...,1037.945847,1001.691705,2039.637552,0.021625,2041.301143,550.681871,-385.7727,1330.253,1716.0257,2591.983014
1,2022-01-01 01:00:00+00:00,384.846011,152.127928,86.865465,15.207601,119.702698,921.140135,5.398335,893.822227,1.637933,...,1040.842833,899.220562,1940.063395,0.022582,1941.723910,639.047005,-433.0126,1253.937,1686.9496,2580.770915
2,2022-01-01 02:00:00+00:00,365.706619,154.854604,86.807320,15.404626,119.564946,835.099673,4.786685,787.981293,1.639639,...,954.664619,792.767978,1747.432597,0.019940,1749.092176,622.773169,-348.4763,1021.750,1370.2263,2371.865345
3,2022-01-01 03:00:00+00:00,379.162111,154.907694,86.882696,15.542090,125.643131,678.922773,2.741542,781.776899,1.634412,...,804.565904,784.518441,1589.084345,0.022718,1590.741475,636.494591,-177.0123,955.643,1132.6553,2227.236066
4,2022-01-01 04:00:00+00:00,370.112134,156.512416,87.018405,14.962279,157.398152,736.866967,2.252833,614.282126,1.630415,...,894.265119,616.534959,1510.800078,0.021027,1512.451520,628.605234,-75.2910,1057.204,1132.4950,2141.056754


In [38]:
#save the dataset now
# save cleaned and feature-engineered settlement dataset
pcs.to_csv("pcs_processed_dk1_utc.csv", index=False)

print("File saved successfully: pcs_processed_dk1_utc.csv")

File saved successfully: pcs_processed_dk1_utc.csv
